# 

In [2]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)

In [5]:
mol_instruction_dataset

DatasetDict({
    description_guided_molecule_design: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 298319
    })
    forward_reaction_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 125384
    })
    molecular_description_generation: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 298319
    })
    property_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 362100
    })
    reagent_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 125384
    })
    retrosynthesis: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 129684
    })
})

In [6]:
forward_data = mol_instruction_dataset['forward_reaction_prediction']

In [8]:
def get_molinst_data_list(
        data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(data)))
    for i in iter_bar:
        data_instance = data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [9]:
list_train_data, list_test_data = get_molinst_data_list(
    data=forward_data,
    instruction_templates=instructions_smol.forward_reaction_prediction,
    task="forward_reaction_prediction",
)

  0%|          | 0/125384 [00:00<?, ?it/s][06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
  0%|          | 95/125384 [00:00<02:12, 948.69it/s][06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighbors
[06:25:10] WARNING: not removing hydrogen atom without neighb

124384 1000


In [11]:
data_dict = {
    "train": list_train_data,
    "test": list_test_data
}


for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_forward_reaction_prediction_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 6393.38 examples/s]
